# ARES-Upgraded Full Training & Evaluation (Colab)

Research-grade pipeline:
1. Mount Drive & install deps
2. Prepare datasets (DIV2K / COCO subsets or user uploads)
3. Deterministic splits
4. Train multi-condition ARES encoder/decoder with curriculum robustness
5. Ablations, rate-distortion, cross-domain, steganalysis
6. Export checkpoints, CSVs, plots, PDF report

**Never fabricate metrics.** All numbers come from measured runs.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ARES_Upgraded'
import os
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)
os.makedirs(f'{DRIVE}/plots', exist_ok=True)
print('Drive ready:', DRIVE)

In [ ]:
# 2. Install dependencies
!pip install -q torch torchvision pillow cryptography scikit-image matplotlib pandas pyyaml tqdm gradio
import torch
print('Torch', torch.__version__, 'CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# 3. Clone / upload ARES-Upgraded source into /content/ARES-Upgraded
# Option A: upload the zip to Drive and extract
import zipfile, shutil
from pathlib import Path
src_zip = Path('/content/drive/MyDrive/ARES_Upgraded/ARES-Upgraded_FINAL.zip')
if src_zip.exists():
    with zipfile.ZipFile(src_zip) as z:
        z.extractall('/content')
    print('Extracted from Drive zip')
else:
    print('Upload ARES-Upgraded source or place zip at', src_zip)
import sys
sys.path.insert(0, '/content/ARES-Upgraded')
print('Python path ready')

In [ ]:
# 4. Dataset preparation (example: download a small public set or use user images)
# For full research use DIV2K + COCO + CelebA with proper splits.
import os, urllib.request
from pathlib import Path
data_root = Path('/content/ares_data')
data_root.mkdir(exist_ok=True)
# Placeholder: user should place real images under data_root/train, val, test, cross_domain
for split in ['train', 'val', 'test', 'cross_domain']:
    (data_root / split).mkdir(exist_ok=True)
print('Place RGB images under', data_root)
print('Then create manifests in data/splits/')

In [ ]:
# 5. Train
!cd /content/ARES-Upgraded && python training/train_ares.py \
  --data /content/ares_data/train \
  --out /content/drive/MyDrive/ARES_Upgraded/checkpoints/ares_upgraded_final.pt \
  --epochs 50 --batch 8 --size 128 --base 32 --lr 1e-4
print('Training finished (or use --pilot for a quick smoke test)')

In [ ]:
# 6. Benchmark (after training)
!cd /content/ARES-Upgraded && python main.py benchmark
print('See results/ for measured CSVs and JSONs')

## Notes
- All metrics are computed from actual encode/decode runs.
- Do not edit test-set results after hyperparameter selection.
- Report Pareto-optimal configurations, not a single "winner".
- Resume from Drive checkpoints after interruptions.